In [ ]:
# from google.colab import runtime
# runtime.unassign()

In [1]:
!rm -rf sample_data/

In [2]:
# dependencies imports and checks

import torch
from torch import nn
import matplotlib.pyplot as plt
import numpy as np
try:
  import mlflow
except:
  %pip install mlflow

try:
  import torchvision
except:
  %pip install torchvision

import torchvision
from torchvision import datasets
from torchvision import models
from torch.utils.data import DataLoader
from torchvision import transforms
from pathlib import Path
print(f"Pytorch version: {torch.__version__}\ntorchvision version:{torchvision.__version__}")

import glob

try:
  import torchinfo
except:
   %pip install torchinfo

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 64.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 70.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.4/228.4 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 123.9/123.9 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [3]:
!pwd

/content


In [4]:
!ls

data_setup.py  engine.py  model.py  train.ipynb  utils.py


In [5]:
# from google.colab import drive
# drive.mount('/content/drive')


In [6]:
# import os
# os.chdir("/content/drive/MyDrive/VLM_pipeline")
# print(os.getcwd())

In [5]:
# Modular imports
from data_setup import create_datasets, get_data
from engine import train
from model import ViT
from utils import (save_model, load_model, display_random_images_from_dataset,
                   set_seed, pred_and_plot_image, plot_loss_acc_curves,
                   plot_random_images_from_path, get_device,
                   walk_through_dir,
                   plot_confusion_matrix,
                   print_patched_image)

In [6]:
from torch.profiler import profile, ProfilerActivity, record_function

In [7]:
device = get_device()
device

'cpu'

In [8]:
get_data()

Did not find /content/data/pizza_steak_sushi directory, creating one...
Unzipping pizza, steak, sushi data...


In [9]:
train_dir = Path('./data/pizza_steak_sushi/train')
test_dir = Path('./data/pizza_steak_sushi/test')

In [17]:
train_images_paths = list((train_dir).glob('*/*.jpg'))
train_images_paths = train_images_paths[:-2]
test_images_paths = list((test_dir).glob('*/*.jpg'))
test_images_paths = test_images_paths[:-2]

In [18]:
import os
for train_image in train_images_paths:
    os.remove(train_image)

for test_image in test_images_paths:
    os.remove(test_image)

In [19]:
walk_through_dir(train_dir)

there are 3 directories and 0 images in inside data/pizza_steak_sushi/train directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/train/steak directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/train/pizza directory
there are 0 directories and 2 images in inside data/pizza_steak_sushi/train/sushi directory


In [20]:
walk_through_dir(test_dir)

there are 3 directories and 0 images in inside data/pizza_steak_sushi/test directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/test/steak directory
there are 0 directories and 0 images in inside data/pizza_steak_sushi/test/pizza directory
there are 0 directories and 2 images in inside data/pizza_steak_sushi/test/sushi directory


In [21]:
training_resolution = 224

vit_preprocess_transformations = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((training_resolution,training_resolution))
])

In [22]:
vit_train_dataloader, vit_test_dataloader, vit_class_names, vit_class_to_idx = create_datasets(
                    train_dir = train_dir,
                    test_dir = test_dir,
                    train_transformations = vit_preprocess_transformations,
                    test_transformations= vit_preprocess_transformations,
                    NUM_WORKERS= 0,
                    BATCH_SIZE = 32,
                    PIN_MEMORY = False,
                    DROP_LAST= True)

In [23]:
set_seed(42)
# reduce regularization for now and try to avoid underfitting over training data
ViT_model = ViT(patch_projection_size=128,
               patch_resolution=16,
               num_patches=196,
               in_channels=3,
               num_transformer_layers=2,
               dropout_rate=0,
               num_heads=4,
               MLP_size=256,
               num_classes = len(vit_class_names))

In [24]:
# Vanilla ViT
from engine import train
from utils import get_device

device = get_device()
NUM_EPOCHS = 40

vit_optimizer = torch.optim.Adam(params = ViT_model.parameters(),
                                 lr=0.001,
                                  betas=(0.9,0.999))

vit_loss_fn = torch.nn.CrossEntropyLoss()

vit_train_results = train(model = ViT_model,
                train_dataloader = vit_train_dataloader,
                test_dataloader = vit_test_dataloader,
                optimizer = vit_optimizer,
                device = device,
                loss_fn = vit_loss_fn,
                epochs = NUM_EPOCHS
                )

  0%|          | 0/40 [00:00<?, ?it/s]

AttributeError: 'int' object has no attribute 'item'

In [ ]:
plot_loss_acc_curves(vit_train_results)


In [ ]:
from utils import plot_confusion_matrix
plot_confusion_matrix(ViT_model,vit_test_dataloader,'cuda',vit_class_names)

In [ ]:
# from datetime import datetime
# model_name = '_epochs_'+ datetime.now().strftime('%Y_Y_%m_m_%d_d_%H:%M:%S')+'_VGG16MINI.pth'
# save_model(model=model,
#            target_dir='/content/models',
#            model_name=model_name)


In [ ]:
# !rm -rf /content/drive/MyDrive/VLM_pipeline/data/pizza_steak_sushi

In [ ]:
from torchvision.datasets import FashionMNIST

train_fashion_mnist = FashionMNIST()